# ForgeLM — DPO Preference Alignment

Align a language model with human preferences using Direct Preference Optimization.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cemililik/ForgeLM/blob/main/notebooks/dpo_alignment.ipynb)

In [ ]:
# Install ForgeLM (clean install to avoid Colab dependency conflicts)
!pip install -q --no-cache-dir git+https://github.com/cemililik/ForgeLM.git
!pip uninstall -y wandb -q  # Remove conflicting wandb (not needed, tensorboard is default)

In [ ]:
config_yaml = """
model:
  name_or_path: "HuggingFaceTB/SmolLM2-1.7B-Instruct"
  max_length: 1024
  load_in_4bit: true

lora:
  r: 16
  alpha: 32
  target_modules: ["q_proj", "v_proj"]

training:
  trainer_type: "dpo"         # Direct Preference Optimization
  dpo_beta: 0.1
  output_dir: "./dpo_checkpoints"
  num_train_epochs: 1
  per_device_train_batch_size: 2
  learning_rate: 5.0e-6       # Lower LR for alignment

data:
  dataset_name_or_path: "argilla/ultrafeedback-binarized-preferences"
  # Dataset must have 'chosen' and 'rejected' columns
"""

with open("dpo_config.yaml", "w") as f:
    f.write(config_yaml)

print("DPO config saved!")

In [ ]:
# Run DPO training
!forgelm --config dpo_config.yaml

## Other Alignment Methods

ForgeLM supports multiple alignment trainers:

| Method | `trainer_type` | Dataset Format | Best For |
|--------|---------------|----------------|----------|
| DPO | `"dpo"` | chosen/rejected pairs | General preference alignment |
| SimPO | `"simpo"` | chosen/rejected pairs | Lower memory (no reference model) |
| KTO | `"kto"` | completion + label (bool) | Binary feedback (production data) |
| ORPO | `"orpo"` | chosen/rejected pairs | SFT + alignment in one stage |
| GRPO | `"grpo"` | prompt only | Reasoning RL (like DeepSeek-R1) |